# Notebook 4: Deployment Strategy and Interactive Demo

## Objectives

This notebook aims to:
1. **Prepare the XGBoost model for production** by training on full dataset (235,795 URLs)
2. **Design the deployment architecture** for the hybrid detection system
3. **Document the Demo Mode approach** and rationale for not using live URL scanning
4. **Create the Streamlit application** with interactive URL testing interface
5. **Implement the violation detection system** showing all warning signs (not just first match)

## Inputs

- **Full Dataset**: dataset4.csv (235,795 URLs × 56 features) - all data for final training
- **Trained Model (Notebook 3)**: XGBoost achieving 99.995% recall from 80/20 split
- **Detection Rules (Notebook 2)**: 6 perfect-precision rules with explanations
- **Business Requirements**: Real-time URL risk assessment, educational visualizations

## Outputs

- **Production Model**: models/xgb_model.pkl (trained on 100% of data, 0.15 MB)
- **Hybrid Detection Function**: `detect_violations()` - checks all 6 rules and returns complete violation profile
- **Streamlit Application**: app.py (346 lines)
  - Search & filter interface (235K URLs from dataset4)
  - Real-time violation detection with bullet-point explanations
  - ML scoring for URLs that pass rules
  - Risk assessment (HIGH/MEDIUM/LOW)
  - Feature importance display
- **Deployment Documentation**: 
  - Demo Mode rationale (focus on ML, not web scraping)
  - Hybrid architecture explanation (Rules → ML)
  - Performance analysis (7x faster than ML-only)
  - False positive analysis (Rule 5 removed - 55 FP found)
- **Requirements File**: requirements.txt (5 packages)
- **Key Design Decision**: Show ALL violations, not just first - more convincing evidence

---

# 4. Deployment Strategy - Interactive Phishing Detection Demo

Building a production-ready demonstration of our hybrid phishing detection system using Streamlit.

## 1. Deployment Overview

**What we're building:**

An interactive phishing detection demo using Streamlit that demonstrates our hybrid approach on dataset4 URLs.

**Demo Mode approach:**
- Select URLs from dataset4 (pre-extracted features)
- Run through hybrid system: Rules → XGBoost
- Show prediction vs actual label
- Display feature breakdown and explanation

**Why Demo Mode (not Live URL scanning)?**

**Live Mode would require:**
1. Fetching live webpages (`requests.get(url)`)
2. Extracting ALL 49 features from raw HTML:
   - Easy features: URLLength, IsHTTPS, NoOfJS (count `<script>` tags)
   - Hard features: TLDLegitimateProb (need TLD reputation database), CharContinuationRate (complex calculation), DomainTitleMatchScore (fuzzy matching)
3. Handling failures: Timeouts, CAPTCHAs, blocked requests, malformed HTML
4. Building 49 feature extractors (significant engineering effort)

**Why we chose Demo Mode:**
- **Focus on ML, not web scraping**: This project demonstrates machine learning for phishing detection, not production web crawling
- **Reliability**: Demo always works (no network issues, timeouts, blocked requests)
- **Assessment-friendly**: Reproducible results for grading
- **Proof of concept**: Shows model performance without production engineering complexity

**In production:** Feature extraction would be handled by a dedicated service (similar to Google Safe Browsing's web crawlers). Our model consumes features, not raw URLs.

**Tech stack:**
- Streamlit (interactive UI)
- XGBoost (pre-trained model from notebook 3)
- pandas (feature handling)

## 2. Model Preparation

**Goal:** Train and save XGBoost model for deployment.

**Why train on full dataset:**

In notebook 3, we used 80/20 split to EVALUATE XGBoost performance (99.995% recall achieved). 

Now for deployment, we retrain on ALL 235,795 URLs to give the model maximum training data. This is standard ML practice:
- **Evaluation phase**: Use split to test performance
- **Deployment phase**: Retrain on all data for strongest model

We are NOT re-testing - we accept the circular validation limitation acknowledged in notebook 3.

**Steps:**
1. Load dataset4 and prepare features (49 numeric features)
2. Train XGBoost on full dataset
3. Save model to `models/xgb_model.pkl`
4. Validate loading works

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from xgboost import XGBClassifier

# Load dataset4
df = pd.read_csv('data/dataset4.csv')

print(f"Total URLs: {len(df):,}")
print(f"Phishing: {(df['label']==0).sum():,} ({(df['label']==0).sum()/len(df)*100:.1f}%)")
print(f"Legitimate: {(df['label']==1).sum():,} ({(df['label']==1).sum()/len(df)*100:.1f}%)")

# Prepare features (same as notebook 3)
features_to_exclude = ['URLSimilarityIndex', 'FILENAME', 'URL', 'Domain', 'TLD', 'Title', 'label']
feature_cols = [col for col in df.columns if col not in features_to_exclude]

X_full = df[feature_cols]
y_full = df['label']

print(f"\nUsing {len(feature_cols)} features for training")

In [ ]:
# Train XGBoost on full dataset
xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

print("Training XGBoost on full dataset (235,795 URLs)...")
xgb_model.fit(X_full, y_full)
print("Training complete!")

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

# Save model
model_path = 'models/xgb_model.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(xgb_model, f)

print(f"\nModel saved to: {model_path}")
print(f"Model file size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

In [ ]:
# Validate: Load model and test prediction
print("Validating model loading...")

with open(model_path, 'rb') as f:
    loaded_model = pickle.load(f)

# Test on first 5 URLs
test_sample = X_full.head(5)
predictions = loaded_model.predict(test_sample)
probabilities = loaded_model.predict_proba(test_sample)

print("\nTest predictions on first 5 URLs:")
for i in range(5):
    actual = "Phishing" if y_full.iloc[i] == 0 else "Legitimate"
    pred = "Phishing" if predictions[i] == 0 else "Legitimate"
    prob_phishing = probabilities[i][0] * 100
    prob_legitimate = probabilities[i][1] * 100
    
    print(f"URL {i+1}: Actual={actual}, Predicted={pred}, Prob(Phishing)={prob_phishing:.1f}%, Prob(Legitimate)={prob_legitimate:.1f}%")

print("\n✓ Model loading and prediction working correctly!")

## 3. Explainable Detection System

**Key insight:** Most phishing sites aren't caught by ONE red flag - they have MULTIPLE obvious violations.

**Why show ALL warning signs (not just first match)?**

1. **More Convincing**: "No HTTPS" vs "5 violations: No HTTPS, Zero Resources, No Trust Signals, IP Domain, Long URL"
2. **Educational**: Users see the full picture of what makes a site suspicious
3. **Better Explainability**: Stakeholders understand WHY the model flagged something
4. **Realistic**: Real phishing sites are lazy - they violate multiple basic practices

**Example: Typical phishing site**
```
URL: http://192.168.1.1/verify-account.php?token=abc123...xyz789

Warning Signs:
- No HTTPS (insecure connection)
- Domain is IP address (not a real domain)
- Long URL (78 characters - obfuscation)
- Zero resources (no JS, CSS, or images)
- No trust signals (no favicon, title, or copyright)

This is obviously phishing (5 violations)
```

vs showing just "No HTTPS" - less convincing!

**Production Flow:**

```
User submits URL
     ↓
┌─────────────────────────────────────┐
│ Stage 1: Violation Detection        │
│ - Check ALL 6 rules                 │
│ - Collect ALL violations            │
│ - Show complete warning profile     │
└─────────────────────────────────────┘
     ↓
┌─────────────────────────────────────┐
│ Decision Engine                     │
│ - If violations found: BLOCK        │
│ - If clean: Proceed to ML           │
└─────────────────────────────────────┘
     ↓ (11.8% of URLs)
┌─────────────────────────────────────┐
│ Stage 2: ML Scoring                 │
│ - XGBoost prediction                │
│ - Probability score                 │
│ - Risk assessment                   │
└─────────────────────────────────────┘
```

**Performance:**
- 88.2% of URLs blocked by rules (instant)
- 11.8% go to ML scoring
- 7x faster than ML-only approach

**The 6 Detection Rules (Zero False Positives):**

1. **Zero Resources**: No JavaScript, CSS, or images (lazy phishing page)
2. **No HTTPS**: Uses HTTP instead of HTTPS (no encryption)
3. **Domain is IP**: Uses IP address instead of domain name
4. **Zero Trust Signals**: No title, favicon, description, or copyright
5. **Excessive Subdomains**: 5+ subdomains (suspicious structure)
6. **Long URL**: >57 characters (obfuscation technique)

Note: "No References" rule was removed after discovering it caused 55 false positives (0.04% of legitimate sites).

### 3.1 Detection Function Implementation

Collect ALL violations for comprehensive explainability:

In [ ]:
def detect_violations(row):
    """
    Check ALL 6 rules and collect ALL violations.
    
    Returns:
        list of (rule_name, explanation) tuples
        Empty list means URL passed all rules (proceed to ML)
    """
    violations = []
    
    # Rule 1: Zero Resources
    if row['NoOfJS'] == 0 and row['NoOfCSS'] == 0 and row['NoOfImage'] == 0:
        violations.append((
            "Zero Resources",
            "Site has no JavaScript, CSS, or images - typical of lazy phishing"
        ))
    
    # Rule 2: No HTTPS
    if row['IsHTTPS'] == 0:
        violations.append((
            "No HTTPS",
            "Site uses HTTP instead of HTTPS - no encryption"
        ))
    
    # Rule 3: Domain is IP
    if row['IsDomainIP'] == 1:
        violations.append((
            "Domain is IP Address",
            "Domain is an IP address instead of proper domain name"
        ))
    
    # Rule 4: Zero Trust Signals
    if (row['HasTitle'] == 0 and row['HasFavicon'] == 0 and 
        row['HasDescription'] == 0 and row['HasCopyrightInfo'] == 0):
        violations.append((
            "Zero Trust Signals",
            "No title, favicon, description, or copyright - minimal effort site"
        ))
    
    # Rule 5: Excessive Subdomains
    if row['NoOfSubDomain'] >= 5:
        violations.append((
            "Excessive Subdomains",
            f"URL has {int(row['NoOfSubDomain'])} subdomains - suspicious structure"
        ))
    
    # Rule 6: Long URL
    if row['URLLength'] > 57:
        violations.append((
            "Long URL",
            f"URL is {int(row['URLLength'])} characters - obfuscation technique"
        ))
    
    return violations


# Test on sample URLs to show violation profiles
print("Testing violation detection on sample URLs:\n")

for i in range(10):
    row = df.iloc[i]
    violations = detect_violations(row)
    actual = "Phishing" if row['label'] == 0 else "Legitimate"
    
    print(f"URL {i+1}: {row['URL'][:60]}...")
    print(f"  Actual Label: {actual}")
    
    if violations:
        print(f"  Violations Found: {len(violations)}")
        for rule, explanation in violations:
            print(f"    - {rule}: {explanation}")
        print(f"  Decision: BLOCK (rule-based)")
    else:
        print(f"  Violations Found: 0")
        print(f"  Decision: Proceed to ML scoring")
    print()

In [ ]:
# Check for false positives: legitimate sites caught by rules
legitimate_sites = df[df['label'] == 1]

print(f"Total legitimate sites: {len(legitimate_sites):,}")

# Apply rules to all legitimate sites
false_positives = []
for idx, row in legitimate_sites.iterrows():
    violations = detect_violations(row)
    if violations:  # Legitimate site has violations - FALSE POSITIVE!
        false_positives.append({
            'URL': row['URL'],
            'violations': violations
        })

print(f"False positives (legitimate sites blocked by rules): {len(false_positives):,}")
print(f"False positive rate: {len(false_positives)/len(legitimate_sites)*100:.4f}%")

if len(false_positives) > 0:
    print(f"\nWARNING: Rules cause {len(false_positives)} false positives!")
    print("\nFirst 10 false positives:")
    for i, fp in enumerate(false_positives[:10]):
        print(f"\n{i+1}. {fp['URL']}")
        print(f"   Violations: {len(fp['violations'])}")
        for rule, explanation in fp['violations']:
            print(f"     - {rule}")
else:
    print("\n✓ PERFECT: Zero false positives! Rules are 100% precise.")
    print("All legitimate sites pass through to ML scoring.")

### 3.2 Verification: Do Rules Cause False Positives?

Critical question: Do any LEGITIMATE sites get caught by these rules?

If yes, we have false positives (blocking good sites). Let's verify:

### 3.2.1 Summary: Rule 5 Removed

**Finding:** The "No References" rule (checking for URLs with zero internal/external links) caused **55 false positives** out of 134,850 legitimate sites (0.04% false positive rate).

**Decision:** Remove Rule 5 to achieve **zero false positives**.

**Final ruleset:** 6 rules with 100% precision
- All 6 remaining rules have zero false positives
- Still achieve high coverage (majority of phishing blocked by rules)
- Trade-off: Slightly more URLs go to ML scoring, but no legitimate sites incorrectly blocked

**Production principle:** Better to let a few more phishing sites through to ML (where they'll still be caught) than to block any legitimate sites with rules.

## 4. Streamlit Interactive Demo

**The complete deployment app is in `app.py`**

### 4.1 App Design Philosophy

**Clean, Educational Interface:**
- Search-driven interaction (no overwhelming 235K URL dropdown)
- Split-view layout: Instructions at top, results at bottom
- Comprehensive violation display (all red flags, not just first match)
- Real-time detection on dataset4 URLs

### 4.2 App Features

**Search & Select:**
- Sidebar search box to filter URLs
- Shows first 10 matching results
- Select any URL to analyze

**Detection Results:**

**Stage 1: Violation Detection**
- Shows ALL violations found (not just first match)
- Bullet-point list with clear explanations
- Example: "5 violations found: No HTTPS, Zero Resources, IP Domain, Long URL, No Trust Signals"
- If violations found → BLOCK decision
- If clean → Proceed to ML

**Stage 2: ML Scoring** (only for URLs that pass rules)
- XGBoost probability scores
- Risk assessment: HIGH (>90%), MEDIUM (70-90%), LOW (<70%)
- Recommendation: Flag / Review / Allow
- Top contributing features

**Educational Components:**
- How-to instructions
- Hybrid system explanation (7 rules listed)
- ML model performance stats
- Actual label comparison (for validation)

### 4.3 Why This Design?

**Search-based (not dropdown):**
- 235,795 URLs would crash a dropdown
- Search is more realistic (users test specific URLs)
- Shows 10 results max (manageable)

**Show all violations (not just first):**
- More convincing evidence
- Educational value
- Matches how humans analyze phishing
- Real phishing sites have multiple red flags

**Split view:**
- Top: Always-visible instructions
- Bottom: Results when URL selected
- Clean default state (no random URL showing)

### 4.4 Running the App

**Launch:**
```bash
streamlit run app.py
```

**Access:**
- http://localhost:8501
- Try searches: "google", ".top", "blogspot", "seedjfly"

**Example Workflow:**
1. Search ".top" → See phishing domains
2. Select a URL
3. View ALL violations (typically 3-5 red flags)
4. See actual label (validate detection)

The app demonstrates explainable AI - showing WHY each URL is flagged.